# Hull Tactical Market Prediction - Modelo Lasso
Notebook basado en Lasso Regression con alpha=0.00001

## Imports

In [ ]:
import os
from pathlib import Path
import datetime
import time

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error

import kaggle_evaluation.default_inference_server

## Project Directory Structure

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Configurations

In [ ]:
# ============ PATHS ============
DATA_PATH = Path('/kaggle/input/hull-tactical-market-prediction/')

# ============ HARDWARE CONFIG ============
os.environ["LOKY_MAX_CPU_COUNT"] = "20"

# ============ RETURNS TO SIGNAL CONFIGS ============
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0
SIGNAL_MULTIPLIER = 20.0  # Equivalente a tu posicion * 20

# ============ MODEL CONFIGS ============
LASSO_ALPHA = 0.00001
RANDOM_STATE = 42

# ============ COLUMNS TO DROP ============
COLS_TO_DROP = ['E7', 'V10']

## Dataclasses

In [ ]:
@dataclass
class DatasetOutput:
    X_train: pd.DataFrame
    X_val: pd.DataFrame
    y_train: pd.Series
    y_val: pd.Series
    feature_cols: list
    imputer: SimpleImputer
    scaler: StandardScaler

@dataclass
class RetToSignalParameters:
    signal_multiplier: float
    min_signal: float = MIN_SIGNAL
    max_signal: float = MAX_SIGNAL

In [ ]:
ret_signal_params = RetToSignalParameters(
    signal_multiplier=SIGNAL_MULTIPLIER
)

## Data Loading Functions

In [ ]:
def load_and_process_train_data():
    """Carga y procesa los datos de entrenamiento"""
    df = pd.read_csv(DATA_PATH / 'train.csv')
    
    # Eliminar columnas especificadas
    df = df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns], errors='ignore')
    
    # Definir target y features
    target_col = 'market_forward_excess_returns'
    exclude_cols = ['date_id', 'forward_returns', 'risk_free_rate', 'market_forward_excess_returns']
    feature_cols = [c for c in df.columns if c not in exclude_cols]
    
    # Eliminar filas con target nulo
    df = df.dropna(subset=[target_col])
    
    X = df[feature_cols]
    y = df[target_col]
    
    return X, y, feature_cols

def load_test_data():
    """Carga los datos de test"""
    return pd.read_csv(DATA_PATH / 'test.csv')

## Preprocessing Functions

In [ ]:
def preprocess_features(X, feature_cols, imputer=None, scaler=None, fit=True):
    """Preprocesa features con imputación y escalado"""
    if fit:
        imputer = SimpleImputer(strategy='mean')
        scaler = StandardScaler()
        
        X_processed = imputer.fit_transform(X)
        X_scaled = scaler.fit_transform(X_processed)
    else:
        X_processed = imputer.transform(X)
        X_scaled = scaler.transform(X_processed)
    
    X_final = pd.DataFrame(X_scaled, columns=feature_cols)
    
    return X_final, imputer, scaler

def split_train_val(X, y, train_ratio=0.8):
    """Divide datos en train y validación"""
    corte = int(len(X) * train_ratio)
    
    X_train = X.iloc[:corte]
    X_val = X.iloc[corte:]
    y_train = y.iloc[:corte]
    y_val = y.iloc[corte:]
    
    return X_train, X_val, y_train, y_val

## Evaluation Functions

In [ ]:
def evaluar_estrategia(y_true, y_pred):
    """Calcula Sharpe Ratio de la estrategia"""
    posicion = np.clip(y_pred * SIGNAL_MULTIPLIER, -1, 1)
    retornos = posicion * y_true
    
    if np.std(retornos) == 0:
        return 0.0
    
    return np.mean(retornos) / np.std(retornos)

def convert_ret_to_signal(ret_arr, params):
    """Convierte predicciones a señales de trading"""
    return np.clip(
        ret_arr * params.signal_multiplier + 1,
        params.min_signal,
        params.max_signal
    )

## Load and Prepare Data

In [ ]:
print("Cargando datos...")
X, y, feature_cols = load_and_process_train_data()

print(f"Forma de X: {X.shape}")
print(f"Número de features: {len(feature_cols)}")
print(f"Forma de y: {y.shape}")

In [ ]:
print("Preprocesando features...")
X_processed, imputer, scaler = preprocess_features(X, feature_cols, fit=True)

print("Dividiendo en train/val...")
X_train, X_val, y_train, y_val = split_train_val(X_processed, y)

print(f"\nEntrenamiento: {len(X_train)} | Validación: {len(X_val)}")

## Train Model

In [ ]:
print("-" * 100)
print(f"{'MODELO':<30} | {'MSE':<10} | {'Sharpe':<10} | {'Volatilidad':<12} | {'Tiempo':<6}")
print("-" * 100)

nombre = f"Lasso (Alpha={LASSO_ALPHA})"
modelo = Lasso(alpha=LASSO_ALPHA, random_state=RANDOM_STATE, max_iter=10000)

t0 = time.time()
modelo.fit(X_train, y_train)
preds = modelo.predict(X_val)
t1 = time.time()

mse = mean_squared_error(y_val, preds)
sharpe = evaluar_estrategia(y_val, preds)
std_preds = np.std(preds)

print(f"{nombre:<30} | {mse:.6f}   | {sharpe:.6f}   | {std_preds:.6f}       | {t1-t0:.2f}s")
print("-" * 100)

## Backtest Visualization

In [ ]:
# Simular estrategia
posicion_simulada = np.sign(preds)
retornos_estrategia = posicion_simulada * y_val

# Calcular retornos acumulados
acumulado_mercado = y_val.cumsum()
acumulado_estrategia = retornos_estrategia.cumsum()

# Crear gráfico
plt.figure(figsize=(12, 6))

plt.plot(acumulado_mercado.reset_index(drop=True), 
         label='Mercado', color='gray', alpha=0.5, linestyle='--')
plt.plot(acumulado_estrategia.reset_index(drop=True), 
         label='Modelo Lasso', color='green', linewidth=2)

plt.title(f'Backtest: Lasso vs Mercado (Sharpe: {sharpe:.4f})')
plt.xlabel('Días de Trading')
plt.ylabel('Retorno Acumulado')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSharpe Ratio: {sharpe:.6f}")
print(f"MSE: {mse:.6f}")
print(f"Retorno acumulado mercado: {acumulado_mercado.iloc[-1]:.6f}")
print(f"Retorno acumulado estrategia: {acumulado_estrategia.iloc[-1]:.6f}")

## Feature Importance

In [ ]:
# Obtener coeficientes del modelo
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': modelo.coef_
})

# Ordenar por valor absoluto
coef_df['abs_coef'] = np.abs(coef_df['coefficient'])
coef_df = coef_df.sort_values('abs_coef', ascending=False)

# Mostrar top 20
print("\nTop 20 Features más importantes:")
print(coef_df.head(20).to_string(index=False))

# Visualizar top 15
top_features = coef_df.head(15)

plt.figure(figsize=(10, 6))
plt.barh(top_features['feature'], top_features['coefficient'])
plt.xlabel('Coeficiente')
plt.title('Top 15 Features - Lasso')
plt.tight_layout()
plt.show()

## Prediction Function for Inference

In [ ]:
def predict(test):
    """Función de predicción para inference server"""
    # Convertir a DataFrame si es necesario
    if not isinstance(test, pd.DataFrame):
        test = pd.DataFrame(test)
    
    # Eliminar columnas especificadas
    test = test.drop(columns=[c for c in COLS_TO_DROP if c in test.columns], errors='ignore')
    
    # Seleccionar solo las features usadas en entrenamiento
    X_test = test[feature_cols]
    
    # Preprocesar
    X_test_processed, _, _ = preprocess_features(
        X_test, feature_cols, imputer=imputer, scaler=scaler, fit=False
    )
    
    # Predecir
    raw_pred = modelo.predict(X_test_processed)[0]
    
    # Convertir a señal
    signal = convert_ret_to_signal(np.array([raw_pred]), ret_signal_params)[0]
    
    return float(signal)

## Inference Server Setup

In [ ]:
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)\n\nif os.getenv('KAGGLE_IS_COMPETITION_RERUN'):\n    inference_server.serve()\nelse:\n    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))